<a href="https://colab.research.google.com/github/eyramg/ds2002-fa26/blob/main/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print("Total revenue:", total_revenue)
print("Total units:", total_units)
print("The 400 orders generated $8,520 in total revenue from 783 units sold.")

Total revenue: 8520.0
Total units: 783
The 400 orders generated $8,520 in total revenue from 783 units sold.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = (
    df.groupby('category')['revenue']
    .sum()
    .reset_index()
)

by_category['share_of_total'] = (
    by_category['revenue'] / total_revenue * 100
)

by_category = by_category.sort_values(
    by='revenue',
    ascending=False
)

by_category

,category,revenue,share_of_total
1,Food,4293.0,50.387324
2,Merch,1771.5,20.792254
0,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
df.columns

Index(['vendor_id', 'category', 'qty', 'price', 'revenue'], dtype='object')

In [5]:
vendor_avg = (
    df.groupby('vendor_id')['revenue']
    .mean()
    .reset_index()
    .sort_values(by='revenue', ascending=False)
)

vendor_avg

,vendor_id,revenue
0,V-01,22.595745
3,V-18,21.750000
1,V-05,20.580645
2,V-10,20.314286


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [6]:
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()

merch_share = merch_revenue / total_revenue * 100

print("Merch revenue:", merch_revenue)
print("Merch share of total revenue:", merch_share)

Merch revenue: 1771.5
Merch share of total revenue: 20.79225352112676


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [7]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North']
})

joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

print("Rows before:", len(df))
print("Rows after:", len(joined))

print("Revenue before:", df['revenue'].sum())
print("Revenue after:", joined['revenue'].sum())

unmatched_vendor = joined.loc[
    joined['vendor_name'].isna(),
    'vendor_id'
].unique()

print("Unmatched vendor:", unmatched_vendor)

Rows before: 400
Rows after: 400
Revenue before: 8520.0
Revenue after: 8520.0
Unmatched vendor: ['V-18']


**The unmatched vendor, and what I did about it:** _..._
The unmatched vendor was V-18. Because the lookup table did not contain a name for this vendor, I kept the vendor_id and retained the order records using a left join. No rows or revenue were removed.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [8]:
pivot_report = pd.pivot_table(
    df,
    values='revenue',
    index='vendor_id',
    columns='category',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

pivot_report

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [9]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a.
The vendors should focus on the categories that generate the most revenue and look for ways to increase sales in the lower-performing categories. Food generated the most revenue at $4,293,  which  was about 50.4% of total revenue, while RainGear generated only $901.50, or about 10.6%. The vendors could continue focusing on Food while finding ways to improve RainGear sales. Overall, the vendors generated $8,520 in revenue from 783 units, so increasing sales in the lower-revenue categories could help improve the overall results.

b.
The least trustworthy answer is probably the vendor comparison of average order revenue because an average can be affected by the number and mix of orders for each vendor. V-01 had the highest average order revenue at about $22.60, but this does not necessarily mean that V-01 generated the most total revenue. Also, the vendor lookup was missing V-18, so some of the vendor information was incomplete. This means the vendor-level results should be interpreted with some caution.